# Chapter 18: Dimensionality Reduction
## Notebook Pendamping Praktik Interaktif

Notebook ini memandu Anda mengeksplorasi teknik reduksi dimensi linier dan non-linier:
1. **Eksplorasi Masalah Kutukan Dimensi** (*Curse of Dimensionality*).
2. **Principal Component Analysis (PCA) dari Dasar & Scikit-Learn**.
3. **Analisis Varians Terjelaskan, Scree Plot & Kriteria Kaiser**.
4. **Rekonstruksi Data & Galat Frobenius** (Inverse Transform).
5. **Singular Value Decomposition (SVD) & Teorema Eckart-Young-Mirsky**.
6. **Pipeline Machine Learning Bebas Kebocoran Data** (*Zero Data Leakage*).
7. **Manifold Learning Non-Linier: t-SNE** (Visualisasi 2D).
8. **Benchmark Komparasi Komprehensif** (PCA vs SVD vs t-SNE vs UMAP).

In [1]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. Memuat dataset telemetri sensor
df = pd.read_csv('data/wireless_sensor_telemetry.csv')
features = [
    'signal_strength_dbm', 'packet_loss_pct', 'latency_ms',
    'temperature_c', 'voltage_v', 'energy_consumption_mah'
]
print(f'Dimensi dataset telemetri: {df.shape[0]} baris x {df.shape[1]} kolom')
print(f'Fitur numerik prediktor: {features}')
print('Distribusi kelas profil sistem:')
print(df['latent_profile'].value_counts())

Dimensi dataset telemetri: 360 baris x 7 kolom
Fitur numerik prediktor: ['signal_strength_dbm', 'packet_loss_pct', 'latency_ms', 'temperature_c', 'voltage_v', 'energy_consumption_mah']
Distribusi kelas profil sistem:
latent_profile
THERMAL_CRITICAL    110
ECO_NORMAL          110
HIGH_LOAD           110
OUTLIER_NOISE        30


### Principal Component Analysis (PCA) & Bobot Pemuatan
PCA memutar sumbu koordinat ke arah varians maksimal data. Nilai singular dan rasio varians terjelaskan untuk masing-masing komponen:

In [2]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[features].values)

pca = PCA(n_components=len(features), random_state=42)
pca.fit(X_scaled)
variances = pca.explained_variance_
ratios = pca.explained_variance_ratio_
cum_ratios = np.cumsum(ratios)

print('Profil Komponen Utama (PCA):')
for i in range(len(features)):
    print(f'PC{i+1}: Varians = {variances[i]:.4f} ({ratios[i]*100:.2f}%) | Kumulatif = {cum_ratios[i]*100:.2f}%')

print('\nMatriks Bobot Pemuatan (Loadings) PC1 dan PC2:')
loadings = pd.DataFrame(pca.components_[:2, :].T, index=features, columns=['PC1', 'PC2'])
print(loadings.round(4))

Profil Komponen Utama (PCA):
PC1: Varians = 4.8438 (80.51%) | Kumulatif = 80.51%
PC2: Varians = 0.4181 (6.95%) | Kumulatif = 87.45%
PC3: Varians = 0.2505 (4.16%) | Kumulatif = 91.62%
PC4: Varians = 0.2181 (3.62%) | Kumulatif = 95.24%
PC5: Varians = 0.1571 (2.61%) | Kumulatif = 97.85%
PC6: Varians = 0.1291 (2.15%) | Kumulatif = 100.00%

Matriks Bobot Pemuatan (Loadings) PC1 dan PC2:
                           PC1     PC2
signal_strength_dbm    -0.3844  0.7620
packet_loss_pct         0.4000  0.5540
latency_ms              0.4212  0.1412
temperature_c           0.4156  0.0279
voltage_v              -0.4081 -0.2172
energy_consumption_mah  0.4189 -0.2109


### Rekonstruksi Data & Galat Norma Frobenius
Menguji inversi proyeksi $\hat{X} = Z W_k^T + \boldsymbol{\mu}$ dari $k=1$ hingga $k=6$.

In [3]:
for k in range(1, len(features) + 1):
    m = PCA(n_components=k, random_state=42)
    Z_k = m.fit_transform(X_scaled)
    X_rec = m.inverse_transform(Z_k)
    diff = X_scaled - X_rec
    mse = np.mean(diff ** 2)
    frob = np.linalg.norm(diff, ord='fro')
    status = ' (Rekonstruksi Sempurna)' if k == len(features) else ''
    print(f'k = {k} | MSE: {mse:.6f} | Norma Frobenius: {frob:.6f}{status}')

k = 1 | MSE: 0.194939 | Norma Frobenius: 20.519951
k = 2 | MSE: 0.125455 | Norma Frobenius: 16.461525
k = 3 | MSE: 0.083816 | Norma Frobenius: 13.455179
k = 4 | MSE: 0.047567 | Norma Frobenius: 10.136256
k = 5 | MSE: 0.021458 | Norma Frobenius: 6.808011
k = 6 | MSE: 0.000000 | Norma Frobenius: 0.000000 (Rekonstruksi Sempurna)


### Dekomposisi SVD & Teorema Eckart-Young-Mirsky
Dekomposisi $X = U \Sigma V^T$ membuktikan aproksimasi rank-$k$ optimal.

In [4]:
U, S, Vt = np.linalg.svd(X_scaled, full_matrices=False)
print(f'Nilai Singular S: {np.round(S, 4)}')
print('Verifikasi Batas Teoretis Eckart-Young-Mirsky:')
for k in [1, 2]:
    X_k = np.dot(U[:, :k] * S[:k], Vt[:k, :])
    emp = np.linalg.norm(X_scaled - X_k, ord='fro')
    theo = np.sqrt(np.sum(S[k:] ** 2))
    print(f'Rank-{k} | Galat Empiris: {emp:.6f} | Teoretis: {theo:.6f}')

Nilai Singular S: [41.7005 12.251   9.4837  8.8486  7.5096  6.808 ]
Verifikasi Batas Teoretis Eckart-Young-Mirsky:
Rank-1 | Galat Empiris: 20.519951 | Teoretis: 20.519951
Rank-2 | Galat Empiris: 16.461525 | Teoretis: 16.461525


### Pipeline Machine Learning Bebas Kebocoran Data
Standarisasi dan PCA hanya dihitung pada data latih (*training set*) sebelum dialirkan ke pengklasifikasi.

In [5]:
X = df[features].values
y = df['latent_profile'].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2, random_state=42)),
    ('clf', LogisticRegression(max_iter=500, random_state=42))
])
pipe.fit(X_train, y_train)
acc = accuracy_score(y_test, pipe.predict(X_test))
print(f'Akurasi model Pipeline (PCA k=2 + Logistic Regression): {acc*100:.2f}%')
print('Kompresi fitur dari 6 dimensi menjadi 2 dimensi berhasil meningkatkan generalisasi dan kecepatan!')

Akurasi model Pipeline (PCA k=2 + Logistic Regression): 96.67%
Kompresi fitur dari 6 dimensi menjadi 2 dimensi berhasil meningkatkan generalisasi dan kecepatan!


### Manifold Learning Non-Linier: t-SNE
Memproyeksikan data ke ruang 2D dengan mempertahankan ketetanggaan probabilitas lokal (distribusi Gaussian di ruang asal dan distribusi t-Student di ruang proyeksi).

In [6]:
tsne = TSNE(n_components=2, perplexity=30, max_iter=1000, random_state=42)
Z_tsne = tsne.fit_transform(X_scaled)
print('t-SNE (perplexity=30) berhasil dikomputasi:')
print(f'Bentuk koordinat proyeksi 2D: {Z_tsne.shape}')
print(f'KL Divergence akhir: {tsne.kl_divergence_:.4f}')

t-SNE (perplexity=30) berhasil dikomputasi:
Bentuk koordinat proyeksi 2D: (360, 2)
KL Divergence akhir: 0.5219
